# Demo simple - Outliers por subpartida

Notebook simple para presentar la tesis: se carga la data desde SQL Server, se trabaja por `NUM_SPN_R`, se generan folds dentro de cada subpartida y se comparan modelos de detección de outliers sobre el valor unitario.

In [ ]:
# =============================================================================
# 1. CONFIGURACION SIMPLE
# =============================================================================

import warnings
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool
from sklearn.model_selection import KFold
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

SEED = 42
SERVIDOR = r"DESKTOP-OGU19A7\SQLEXPRESS,56878"
BASE_DATOS = "DB_GEE_DW_ADUANAS"
ESQUEMA = "SC_ADUANA"
PROCEDIMIENTO = "SP_VALORES_UNITARIOS"
DRIVER = "ODBC+Driver+17+for+SQL+Server"

FEC_INI = "2024-01-01"
FEC_FIN = "2024-12-31"

MIN_REGISTROS = 36
N_SUBPARTIDAS = 5
N_FOLDS = 5
CONTAMINACION = 0.05

print("Configuracion cargada")
print(f"Servidor  : {SERVIDOR}")
print(f"BD        : {BASE_DATOS}")
print(f"SP        : {ESQUEMA}.{PROCEDIMIENTO}")
print(f"Periodo   : {FEC_INI} a {FEC_FIN}")

In [ ]:
# =============================================================================
# 2. CARGA DIRECTA DESDE SQL SERVER
# =============================================================================

inicio = time.perf_counter()

url = (
    f"mssql+pyodbc://{SERVIDOR}/{BASE_DATOS}"
    f"?driver={DRIVER}&Trusted_Connection=yes"
)

motor = create_engine(url, echo=False, poolclass=NullPool)

try:
    sentencia = text(
        f"EXEC [{BASE_DATOS}].[{ESQUEMA}].[{PROCEDIMIENTO}] "
        f"@ACCION='EDA_BASE', "
        f"@FEC_INI='{FEC_INI}', "
        f"@FEC_FIN='{FEC_FIN}'"
    )

    with motor.connect() as conn:
        resultado = conn.execute(sentencia)
        df = pd.DataFrame.from_records(resultado.fetchall(), columns=list(resultado.keys()))
finally:
    motor.dispose()

if df.empty:
    raise ValueError("El SP retorno 0 registros. Revisa la ingesta o el rango de fechas.")

# Tipos simples segun lo que devuelve EDA_BASE.
df["NUM_SPN_R"] = df["NUM_SPN_R"].astype(str)
df["ANIO_C"] = pd.to_numeric(df["ANIO_C"], errors="coerce").astype("Int64")
df["MTO_VALOR_UNTARIO_V"] = pd.to_numeric(df["MTO_VALOR_UNTARIO_V"], errors="coerce")
df["FOB_DOLAR"] = pd.to_numeric(df["FOB_DOLAR"], errors="coerce")
df["PESO_NETO"] = pd.to_numeric(df["PESO_NETO"], errors="coerce")

# Limpieza minima.
df = df[
    df["NUM_SPN_R"].notna()
    & df["ANIO_C"].notna()
    & df["MTO_VALOR_UNTARIO_V"].notna()
    & df["FOB_DOLAR"].notna()
    & df["PESO_NETO"].notna()
    & (df["MTO_VALOR_UNTARIO_V"] > 0)
    & (df["FOB_DOLAR"] > 0)
    & (df["PESO_NETO"] > 0)
].copy()

df["LOG_VALOR_UNITARIO"] = np.log1p(df["MTO_VALOR_UNTARIO_V"])
df["ID_REGISTRO"] = np.arange(1, len(df) + 1)

print(f"Registros : {len(df):,}")
print(f"Partidas  : {df['NUM_SPN_R'].nunique():,}")
print(f"Anios     : {sorted(df['ANIO_C'].dropna().unique().tolist())}")
print(f"Columnas  : {list(df.columns)}")
print(f"Tiempo    : {time.perf_counter() - inicio:.2f} segundos")

df.head()

In [ ]:
# =============================================================================
# 3. SELECCION SIMPLE DE SUBPARTIDAS PARA LA DEMO
# =============================================================================

conteo = (
    df.groupby("NUM_SPN_R")
    .size()
    .reset_index(name="REGISTROS")
    .sort_values("REGISTROS", ascending=False)
)

subpartidas_demo = conteo[conteo["REGISTROS"] >= MIN_REGISTROS].head(N_SUBPARTIDAS)["NUM_SPN_R"].tolist()

df_demo = df[df["NUM_SPN_R"].isin(subpartidas_demo)].copy()

print("Subpartidas demo:")
print(subpartidas_demo)
print(f"Registros demo: {len(df_demo):,}")

conteo.head(10)

In [ ]:
# =============================================================================
# 4. FOLDS DENTRO DE CADA SUBPARTIDA
# =============================================================================

folds = []

for subpartida, df_sub in df_demo.groupby("NUM_SPN_R"):
    df_sub = df_sub.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_splits = min(N_FOLDS, len(df_sub))

    if n_splits < 2:
        continue

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for fold_id, (idx_train, idx_test) in enumerate(kfold.split(df_sub), start=1):
        folds.append({
            "subpartida": subpartida,
            "fold": fold_id,
            "train": df_sub.iloc[idx_train].copy(),
            "test": df_sub.iloc[idx_test].copy(),
        })

if not folds:
    raise ValueError("No se crearon folds. Baja MIN_REGISTROS o revisa la data.")

resumen_folds = pd.DataFrame([
    {
        "subpartida": f["subpartida"],
        "fold": f["fold"],
        "n_train": len(f["train"]),
        "n_test": len(f["test"]),
    }
    for f in folds
])

resumen_folds.head(10)

In [ ]:
# =============================================================================
# 5. VALIDACION SIMPLE CON OUTLIERS SINTETICOS
# =============================================================================

resultados = []

for item in folds:
    subpartida = item["subpartida"]
    fold_id = item["fold"]
    train = item["train"].copy()
    test = item["test"].copy()

    n_sint = max(2, int(len(test) * CONTAMINACION))
    sint = test.sample(n=min(n_sint, len(test)), replace=True, random_state=SEED).copy()

    mediana_vu = train["MTO_VALOR_UNTARIO_V"].median()
    mitad = len(sint) // 2
    sint["MTO_VALOR_UNTARIO_V"] = mediana_vu
    sint.iloc[:mitad, sint.columns.get_loc("MTO_VALOR_UNTARIO_V")] = mediana_vu * 0.08
    sint.iloc[mitad:, sint.columns.get_loc("MTO_VALOR_UNTARIO_V")] = mediana_vu * 6.00
    sint["LOG_VALOR_UNITARIO"] = np.log1p(sint["MTO_VALOR_UNTARIO_V"])

    test["y_true"] = 0
    sint["y_true"] = 1
    test_eval = pd.concat([test, sint], ignore_index=True)

    # Modelo 1: IQR por subpartida.
    q1 = train["MTO_VALOR_UNTARIO_V"].quantile(0.25)
    q3 = train["MTO_VALOR_UNTARIO_V"].quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr
    pred_iqr = ((test_eval["MTO_VALOR_UNTARIO_V"] < li) | (test_eval["MTO_VALOR_UNTARIO_V"] > ls)).astype(int)

    resultados.append({
        "subpartida": subpartida,
        "fold": fold_id,
        "modelo": "IQR",
        "precision": precision_score(test_eval["y_true"], pred_iqr, zero_division=0),
        "recall": recall_score(test_eval["y_true"], pred_iqr, zero_division=0),
        "f1": f1_score(test_eval["y_true"], pred_iqr, zero_division=0),
    })

    # Modelo 2: Isolation Forest.
    iso = IsolationForest(contamination=CONTAMINACION, random_state=SEED)
    iso.fit(train[["LOG_VALOR_UNITARIO"]])
    pred_iso = (iso.predict(test_eval[["LOG_VALOR_UNITARIO"]]) == -1).astype(int)

    resultados.append({
        "subpartida": subpartida,
        "fold": fold_id,
        "modelo": "IsolationForest",
        "precision": precision_score(test_eval["y_true"], pred_iso, zero_division=0),
        "recall": recall_score(test_eval["y_true"], pred_iso, zero_division=0),
        "f1": f1_score(test_eval["y_true"], pred_iso, zero_division=0),
    })

    # Modelo 3: LOF novelty.
    n_neighbors = min(20, max(2, len(train) - 1))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=CONTAMINACION, novelty=True)
    lof.fit(train[["LOG_VALOR_UNITARIO"]])
    pred_lof = (lof.predict(test_eval[["LOG_VALOR_UNITARIO"]]) == -1).astype(int)

    resultados.append({
        "subpartida": subpartida,
        "fold": fold_id,
        "modelo": "LOF",
        "precision": precision_score(test_eval["y_true"], pred_lof, zero_division=0),
        "recall": recall_score(test_eval["y_true"], pred_lof, zero_division=0),
        "f1": f1_score(test_eval["y_true"], pred_lof, zero_division=0),
    })

tabla_resultados = pd.DataFrame(resultados)

tabla_resumen = (
    tabla_resultados
    .groupby(["subpartida", "modelo"], as_index=False)
    .agg(
        precision_prom=("precision", "mean"),
        recall_prom=("recall", "mean"),
        f1_prom=("f1", "mean"),
    )
    .sort_values(["subpartida", "f1_prom"], ascending=[True, False])
)

tabla_resumen

In [ ]:
# =============================================================================
# 6. MODELO GANADOR POR SUBPARTIDA
# =============================================================================

ganadores = (
    tabla_resumen
    .sort_values(["subpartida", "f1_prom"], ascending=[True, False])
    .groupby("subpartida")
    .head(1)
    .reset_index(drop=True)
)

ganadores

In [ ]:
# =============================================================================
# 7. ALERTAS REALES SIMPLES CON IQR POR SUBPARTIDA
# =============================================================================

alertas = []

for subpartida, df_sub in df_demo.groupby("NUM_SPN_R"):
    q1 = df_sub["MTO_VALOR_UNTARIO_V"].quantile(0.25)
    q3 = df_sub["MTO_VALOR_UNTARIO_V"].quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr

    tmp = df_sub.copy()
    tmp["LIMITE_INFERIOR"] = li
    tmp["LIMITE_SUPERIOR"] = ls
    tmp["ES_ALERTA"] = ((tmp["MTO_VALOR_UNTARIO_V"] < li) | (tmp["MTO_VALOR_UNTARIO_V"] > ls)).astype(int)
    alertas.append(tmp[tmp["ES_ALERTA"] == 1])

alertas_reales = pd.concat(alertas, ignore_index=True) if alertas else pd.DataFrame()

print(f"Alertas reales detectadas: {len(alertas_reales):,}")

alertas_reales[[
    "ID_REGISTRO",
    "NUM_SPN_R",
    "ANIO_C",
    "MTO_VALOR_UNTARIO_V",
    "FOB_DOLAR",
    "PESO_NETO",
    "SECTOR",
    "TIPO_PRODUCTO",
    "ADUANA",
    "LIMITE_INFERIOR",
    "LIMITE_SUPERIOR",
]].head(20)

In [ ]:
# =============================================================================
# 8. GRAFICO SIMPLE PARA LA PRESENTACION
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 5))

for subpartida, df_sub in df_demo.groupby("NUM_SPN_R"):
    ax.scatter(
        [subpartida] * len(df_sub),
        df_sub["MTO_VALOR_UNTARIO_V"],
        alpha=0.35,
        s=15,
    )

if not alertas_reales.empty:
    ax.scatter(
        alertas_reales["NUM_SPN_R"],
        alertas_reales["MTO_VALOR_UNTARIO_V"],
        s=45,
        marker="x",
        label="Alerta IQR",
    )

ax.set_yscale("log")
ax.set_title("Valor unitario por subpartida: cada subpartida es un universo")
ax.set_xlabel("Subpartida")
ax.set_ylabel("Valor unitario FOB / Peso neto")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## Lectura para exponer

1. Primero se carga la data del procedimiento `EDA_BASE`.
2. La unidad de análisis es `NUM_SPN_R`; no se mezclan subpartidas.
3. Los folds se crean dentro de cada subpartida.
4. Se insertan outliers sintéticos solo para medir si el modelo los recupera.
5. En datos reales, las alertas no significan fraude; son casos priorizados para revisión técnica.